# [V3] Huan luyen YOLOv8s-P2 - Zalo AI Traffic Sign 2020

Ban V3 nang cap tu ban goc voi 3 thay doi chinh:

| Hang muc | Ban cu | Ban V3 |
|---|---|---|
| Chia du lieu | 80% Train / 20% Val (khong co tap Test) | **70 / 10 / 20** co tap Hold-out Test rieng |
| So epoch | 50 (chay cung so, khong biet da hoi tu chua) | **100 + Early Stopping `patience=15`** |
| Nhat ky | Chi co `results.csv` tho | Xuat them `yolov8_training_history.json` de ve Learning Curve |

**Moi truong:** Kaggle Notebook, GPU P100 hoac T4.
**Dataset can Add:** `phhasian0710/za-traffic-2020`.

> ### Nguyen tac vang cua phien ban V3
>
> Toan bo du lieu duoc chia lai theo ty le **70% Train / 10% Validation / 20% Hold-out Test** bang `split_dataset.py` voi `random_seed=42`.
>
> - Thu muc `holdout_test/` **khong duoc khai bao trong `data.yaml`** va tuyet doi khong dung o bat ky buoc nao trong notebook nay.
> - Ca 3 model deu goi cung mot script chia, cung mot seed, nen chac chan dung chung mot tap Test.
> - Chi mo tap Hold-out ra dung mot lan duy nhat o notebook `evaluate_3_models.ipynb`.

## Cell 0: Cai dat thu vien va kiem tra GPU

Moi truong Kaggle khong co san `ultralytics`, phai tu cai. Dat cell nay len dau tien de neu thieu mang hay het han muc thi notebook bao loi ngay tu buoc dau, thay vi chay chia du lieu xong moi chet o cell train.

**Luu y chon GPU:** phai chon **T4 x2**, khong duoc chon **P100**. Ban PyTorch trong moi truong Kaggle hien tai chi bien dich cho kien truc tu sm_70 tro len, ma P100 la Pascal sm_60 nen se bao loi `CUDA error: no kernel image is available for execution on the device`. T4 la sm_75 nen chay binh thuong, lai co Tensor Core nen train AMP con nhanh hon P100.

In [ ]:
!pip install -q ultralytics

# Lenh pip o tren that bai thi notebook van chay tiep binh thuong, nen phai import
# ngay tai day de neu thieu thu vien thi dung han o cell dau chu khong keo dai them.
import ultralytics

ultralytics.checks()

import torch

print(f"\nGPU           : {torch.cuda.get_device_name(0)}")
print(f"Compute cap   : sm_{''.join(map(str, torch.cuda.get_device_capability(0)))}")

# Thu mot phep tinh nho tren GPU de bat som truong hop chon nham P100 (sm_60).
try:
    thu = torch.randn(100, 100, device='cuda')
    _ = (thu @ thu).sum().item()
    print("GPU chay duoc binh thuong.")
except Exception as loi:
    raise RuntimeError(
        f"GPU dang chon khong tuong thich voi ban PyTorch cua moi truong ({loi}).\n"
        "Cach xu ly: o thanh ben phai, muc Accelerator doi tu 'GPU P100' sang 'GPU T4 x2'."
    )

## Cell 1: Lay script chia du lieu dung chung

Ca 3 model deu dung chung file `split_dataset.py` thay vi moi notebook tu viet mot doan chia rieng. Lam vay moi dam bao 3 model duoc chia y het nhau — neu copy-paste 3 lan thi chi can lech mot dong la ca bang so sanh mat gia tri.

In [ ]:
import os
import urllib.request

# Keo file split_dataset.py tu GitHub ve de dung chung mot phep chia voi 2 model kia.
URL_SCRIPT = ('https://raw.githubusercontent.com/vtdung23/Object-Detection-Application/main/Traffic-Sign-Detection-ZaloAI/data_preparation/split_dataset.py')

if not os.path.exists('split_dataset.py'):
    try:
        urllib.request.urlretrieve(URL_SCRIPT, 'split_dataset.py')
        print("Da tai split_dataset.py tu GitHub ve.")
    except Exception as loi:
        raise FileNotFoundError(
            f"Khong tai duoc split_dataset.py ({loi}).\n"
            "Cach khac: mo file split_dataset.py trong repo, copy noi dung roi tao thu cong "
            "file cung ten o thu muc lam viec hien tai."
        )
else:
    print("File split_dataset.py da co san.")

## Cell 2: Chay chia du lieu 70-10-20

Script se tu do tim dataset trong `/kaggle/input/`, chia bang seed 42 roi ghi ra:
- `/kaggle/working/data_v3/dataset_train_val/` — thu muc duy nhat duoc phep dung de train.
- `/kaggle/working/data_v3/holdout_test/` — tap an, khong dung o notebook nay.

In [ ]:
!python split_dataset.py --output-root /kaggle/working/data_v3

DATA_YAML = '/kaggle/working/data_v3/dataset_train_val/data.yaml'
print("\nNoi dung file data.yaml se dung de train:")
print(open(DATA_YAML, encoding='utf-8').read())

## Cell 3: Dung kien truc YOLOv8s-P2

Giu nguyen kien truc P2 cua ban goc: them nhanh P2 (Stride 4) de bat vat the sieu nho, phat hien o 4 cap do P2/P3/P4/P5 thay vi 3 cap mac dinh.

In [ ]:
p2_yaml_content = """
# Ultralytics YOLO, AGPL-3.0 license
# YOLOv8-p2 architecture
nc: 7  # number of classes
scales:
  s: [0.33, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]  # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]  # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]  # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]  # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]  # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]  # cat backbone P4
  - [-1, 3, C2f, [512]]  # 12

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]  # cat backbone P3
  - [-1, 3, C2f, [256]]  # 15 (P3/8-small)

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]  # cat backbone P2
  - [-1, 3, C2f, [128]]  # 18 (P2/4-xsmall)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]  # cat head P3
  - [-1, 3, C2f, [256]]  # 21 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]  # cat head P4
  - [-1, 3, C2f, [512]]  # 24 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]  # cat head P5
  - [-1, 3, C2f, [1024]]  # 27 (P5/32-large)

  - [[18, 21, 24, 27], 1, Detect, [nc]]  # Detect(P2, P3, P4, P5)
"""

with open('/kaggle/working/yolov8s-p2.yaml', 'w', encoding='utf-8') as f:
    f.write(p2_yaml_content)
print("Da tao kien truc YOLOv8s-P2.")

## Cell 4: Huan luyen (100 epochs + Early Stopping)

Toan bo sieu tham so tu EDA duoc giu nguyen (`imgsz=1280`, `cls=2.0`, `mosaic`, `translate=0.2`...). Co 3 thay doi:

- `epochs=50` → `epochs=100`: cho model du dat de hoi tu.
- Them `patience=15`: neu 15 epoch lien tiep ma diem fitness tren tap Val khong tang, Ultralytics tu dong dung va giu lai `best.pt` cua epoch tot nhat.
- Bo `fl_gamma=2.0`: xem giai thich ben duoi.

**Ve tham so `fl_gamma` da bi go bo.** Ban goc truyen `fl_gamma=2.0` voi y dinh bat Focal Loss de tri mat can bang du lieu. Thuc te khoa nay la di san tu file cau hinh cua YOLOv5, Ultralytics co giu ten trong danh sach tham so mot thoi gian nhung **khong he noi no vao ham loss cua YOLOv8** — `v8DetectionLoss` dung `BCEWithLogitsLoss` cho nhanh phan loai, khong doc `gamma` o dau ca. Den cac ban gan day thi Ultralytics xoa han khoa nay, truyen vao se bao `SyntaxError: 'fl_gamma' is not a valid YOLO argument`.

Nghia la ke ca ban goc chay tron ven thi Focal Loss cung chua bao gio duoc kich hoat. Bo dong nay di **khong lam thay doi ket qua huan luyen**, chi lam cho code khop voi thuc te. Nhiem vu chong mat can bang du lieu hien do `cls=2.0` dam nhiem: he so nay nhan doi phan dong gop cua loss phan loai so voi loss hoi quy khung, ep model uu tien sua loi doc nham loai bien bao. Theo dung ket luan trong phan EDA, muc mat can bang cua bo du lieu nay chi khoang 1:5.5 (Moderate Imbalance) nen bien phap nay la du.

Y nghia cua Early Stopping o day khong phai de tiet kiem thoi gian, ma de giai quyet chuyen **CNN va Transformer hoi tu o toc do rat khac nhau**. Neu ep ca 3 model chay cung 50 epoch thi model nao hoi tu cham se bi cat ngang luc chua chin, con model hoi tu nhanh thi thua ra hang chuc epoch overfitting. Cho ca 3 cung tran 100 epoch roi de Early Stopping tu quyet dinh diem dung moi la so sanh cong bang.

In [ ]:
from ultralytics import YOLO

SO_EPOCHS = 100
PATIENCE = 15

model = YOLO('/kaggle/working/yolov8s-p2.yaml')
model.load('yolov8s.pt')  # nap trong so pretrained cho hoi tu nhanh

results = model.train(
    data=DATA_YAML,          # chi tro toi train + val, khong he biet toi tap Test
    epochs=SO_EPOCHS,
    patience=PATIENCE,       # [V3] Early Stopping
    imgsz=1280,              # [E3] High-res bao toan pixel vat the nho
    batch=8,
    max_det=50,              # [E1, E2] Toi uu luong NMS
    iou=0.6,                 # [E2] Giu bien bao dung canh nhau
    optimizer='AdamW',
    cos_lr=True,             # Cosine Annealing

    # --- Ky thuat Ham Loss ---
    # Ban cu co them fl_gamma=2.0 nhung da phai bo. Ultralytics ke thua khoa nay tu
    # thoi YOLOv5 va khong he noi no vao ham loss cua YOLOv8, den cac ban gan day thi
    # xoa han khoi danh sach tham so hop le. Viec tri mat can bang du lieu bay gio dua
    # hoan toan vao he so cls duoi day.
    cls=2.0,                 # [E1, M4.1] Tang cls_gain gap doi box_gain
    box=1.0,

    # --- Augmentation & Khac phuc Center Bias ---
    mosaic=1.0,
    degrees=10.0,
    translate=0.2,           # [E4] Random Shift pha Center Bias

    project='/kaggle/working/zalo_traffic',
    name='yolov8s_p2_v3',
    device=0,
)

## Cell 5: Ham xuat nhat ky huan luyen ra JSON

Ultralytics ghi san file `results.csv` sau moi epoch. Ham duoi doc file do va chuyen thanh JSON gon gang de dung ve bieu do trong bao cao.

Mot luu y ky thuat: YOLOv8 va RT-DETR co ten cot loss khac nhau (YOLO dung `box/cls/dfl`, RT-DETR dung `giou/cls/l1`), nen ham do cot theo tu khoa `loss` thay vi go cung ten cot. Nho vay dung chung duoc mot ham cho ca hai model.

In [ ]:
import csv
import json
import os


def _ep_kieu_so(gia_tri):
    """Ep chuoi trong file CSV ve so thuc. Tra ve None neu o do bi trong."""
    try:
        return float(gia_tri)
    except (TypeError, ValueError):
        return None


def _do_cot_map(ten_cac_cot):
    """Do ten cot chua mAP@50 va mAP@50-95 trong file results.csv."""
    cot_map50 = None
    cot_map5095 = None
    for ten_cot in ten_cac_cot:
        ten_thuong = ten_cot.lower()
        if 'map50-95' in ten_thuong:
            cot_map5095 = ten_cot
        elif 'map50' in ten_thuong:
            cot_map50 = ten_cot
    return cot_map50, cot_map5095


def xuat_lich_su_ultralytics(thu_muc_ket_qua, ten_model, duong_dan_luu,
                             epochs_du_kien=None, patience=None):
    """Doc results.csv cua Ultralytics roi ghi ra file JSON lich su huan luyen.

    Moi epoch lay 5 truong: epoch_id, train_loss, val_loss, mAP_50, mAP_50_95.
    Loss tong = cong don tat ca cac cot co chu 'loss' (box + cls + dfl, hoac giou + cls + l1).
    """
    duong_dan_csv = os.path.join(thu_muc_ket_qua, 'results.csv')
    if not os.path.exists(duong_dan_csv):
        raise FileNotFoundError(f"Khong thay {duong_dan_csv}. Model da train xong chua?")

    lich_su = []
    with open(duong_dan_csv, 'r', encoding='utf-8') as f:
        bo_doc = csv.DictReader(f)
        ten_cac_cot = [c.strip() for c in bo_doc.fieldnames]
        cot_map50, cot_map5095 = _do_cot_map(ten_cac_cot)

        for so_thu_tu, dong in enumerate(bo_doc, start=1):
            dong = {k.strip(): v for k, v in dong.items() if k}

            # Cong don cac thanh phan loss cua tap train va tap val
            tong_loss_train = 0.0
            tong_loss_val = 0.0
            for ten_cot, gia_tri in dong.items():
                so = _ep_kieu_so(gia_tri)
                if so is None or 'loss' not in ten_cot.lower():
                    continue
                if ten_cot.lower().startswith('train'):
                    tong_loss_train += so
                elif ten_cot.lower().startswith('val'):
                    tong_loss_val += so

            epoch_id = _ep_kieu_so(dong.get('epoch')) or so_thu_tu

            lich_su.append({
                'epoch_id': int(epoch_id),
                'train_loss': round(tong_loss_train, 6),
                'val_loss': round(tong_loss_val, 6),
                'mAP_50': round(_ep_kieu_so(dong.get(cot_map50)) or 0.0, 6),
                'mAP_50_95': round(_ep_kieu_so(dong.get(cot_map5095)) or 0.0, 6),
            })

    ket_qua = {
        'model': ten_model,
        'epochs_du_kien': epochs_du_kien,
        'epochs_thuc_te': len(lich_su),
        'early_stopping_patience': patience,
        'nguon_du_lieu': duong_dan_csv,
        'history': lich_su,
    }

    os.makedirs(os.path.dirname(duong_dan_luu) or '.', exist_ok=True)
    with open(duong_dan_luu, 'w', encoding='utf-8') as f:
        json.dump(ket_qua, f, ensure_ascii=False, indent=2)

    print(f"Da ghi lich su huan luyen: {duong_dan_luu}")
    print(f"  Du kien {epochs_du_kien} epochs, thuc te chay {len(lich_su)} epochs.")
    if epochs_du_kien and len(lich_su) < epochs_du_kien:
        print("  => Early Stopping da kich hoat, model dung som vi khong con cai thien.")
    return ket_qua

## Cell 6: Sinh file `yolov8_training_history.json`

In [ ]:
THU_MUC_KET_QUA = '/kaggle/working/zalo_traffic/yolov8s_p2_v3'
DUONG_DAN_JSON = '/kaggle/working/yolov8_training_history.json'

lich_su_yolo = xuat_lich_su_ultralytics(
    thu_muc_ket_qua=THU_MUC_KET_QUA,
    ten_model='YOLOv8s-P2',
    duong_dan_luu=DUONG_DAN_JSON,
    epochs_du_kien=SO_EPOCHS,
    patience=PATIENCE,
)

# In thu 3 epoch dau cho de kiem tra
for moc in lich_su_yolo['history'][:3]:
    print(moc)

## Cell 7: Ve Learning Curve

Bieu do nay dung de tra loi 2 cau hoi trong bao cao: model da hoi tu that chua, va co bi overfitting khong (dau hieu: `val_loss` quay dau di len trong khi `train_loss` van giam).

In [ ]:
import matplotlib.pyplot as plt


def ve_learning_curve(ket_qua_lich_su, duong_dan_luu_anh):
    """Ve 2 do thi canh nhau: duong cong Loss va duong cong mAP theo tung epoch."""
    lich_su = ket_qua_lich_su['history']
    cac_epoch = [m['epoch_id'] for m in lich_su]

    fig, (truc_trai, truc_phai) = plt.subplots(1, 2, figsize=(14, 5))

    # Do thi 1: Loss - dung de phat hien Overfitting (val_loss quay dau di len)
    truc_trai.plot(cac_epoch, [m['train_loss'] for m in lich_su], label='Train Loss')
    truc_trai.plot(cac_epoch, [m['val_loss'] for m in lich_su], label='Val Loss')
    truc_trai.set_xlabel('Epoch')
    truc_trai.set_ylabel('Loss')
    truc_trai.set_title(f"Learning Curve - {ket_qua_lich_su['model']}")
    truc_trai.legend()
    truc_trai.grid(alpha=0.3)

    # Do thi 2: mAP - dung de xac nhan diem hoi tu that su
    truc_phai.plot(cac_epoch, [m['mAP_50'] for m in lich_su], label='mAP@50')
    truc_phai.plot(cac_epoch, [m['mAP_50_95'] for m in lich_su], label='mAP@50-95')
    truc_phai.set_xlabel('Epoch')
    truc_phai.set_ylabel('mAP')
    truc_phai.set_title('Duong cong do chinh xac')
    truc_phai.legend()
    truc_phai.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(duong_dan_luu_anh, dpi=130)
    plt.show()
    print(f"Da luu bieu do: {duong_dan_luu_anh}")

In [ ]:
ve_learning_curve(lich_su_yolo, '/kaggle/working/yolov8_learning_curve.png')

## Cell 8: Nen ket qua de tai ve

File `.zip` gom trong so, `results.csv`, file JSON lich su va bieu do Learning Curve.

In [ ]:
!cp /kaggle/working/yolov8_training_history.json /kaggle/working/zalo_traffic/yolov8s_p2_v3/
!cp /kaggle/working/yolov8_learning_curve.png /kaggle/working/zalo_traffic/yolov8s_p2_v3/
!zip -r -q /kaggle/working/yolov8_v3_results.zip /kaggle/working/zalo_traffic
print("Da nen xong. Tai file yolov8_v3_results.zip o cot Output ben phai.")